In [8]:
# %%
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import re
from datetime import datetime, timedelta

# =========================================
# CONFIG
# =========================================

UNDERLYING_MAP = {
    "PETR": "PETR4",
    "VALE": "VALE3",
    "BBDC": "BBDC4",
    "BBAS": "BBAS3",
    "ABEV": "ABEV3",
    "WEGE": "WEGE3",
    "B3SA": "B3SA3",
    "SUZB": "SUZB3",
}

UNDERLYINGS_BASE = list(UNDERLYING_MAP.keys())

# ticker válido de opção B3
OPTION_REGEX = re.compile(r"^[A-Z]{4}[A-Z][0-9]{2,3}$")


# =========================================
# FUNÇÃO: BAIXAR ÚLTIMO COTAHIST DISPONÍVEL
# =========================================

def download_latest_cotahist(max_lookback_days=10):

    base_url = "https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_D{date}.ZIP"
    today = datetime.today()

    for i in range(max_lookback_days):

        ref = today - timedelta(days=i)
        date_str = ref.strftime("%d%m%Y")
        url = base_url.format(date=date_str)

        try:

            r = requests.get(url, timeout=20)

            if r.status_code != 200:
                continue

            z = zipfile.ZipFile(io.BytesIO(r.content))
            file_name = z.namelist()[0]
            data = z.read(file_name).decode("latin1")
            lines = data.splitlines()

            print("✅ Arquivo encontrado:", url)
            print("Arquivo interno:", file_name)
            print("Total de linhas:", len(lines))

            return ref.date(), lines

        except Exception:
            continue

    raise RuntimeError("❌ Não foi possível baixar um COTAHIST recente.")


# =========================================
# FUNÇÃO: PARSE DO COTAHIST
# =========================================

def parse_cotahist(lines, underlyings_base):

    records = []

    for l in lines:

        if not l.startswith("01"):
            continue

        ticker = l[12:24].strip().upper()

        if not any(ticker.startswith(u) for u in underlyings_base):
            continue

        close = int(l[108:121]) / 100
        volume = int(l[152:170])

        strike_raw = l[188:201]
        expiration_raw = l[202:210]

        strike = np.nan
        expiration = pd.NaT


        # =====================================
        # STRIKE
        # =====================================

        if strike_raw.strip().isdigit():

            strike_val = int(strike_raw)

            if strike_val != 0:

                strike = strike_val / 100

                # REMOVE STRIKES TERMINADOS EM .0
                # elimina PETRD540, PETRD550 etc
                if strike.is_integer():
                    strike = np.nan


        # =====================================
        # EXPIRATION
        # =====================================

        if expiration_raw.strip().isdigit():

            exp_val = int(expiration_raw)

            if exp_val != 0:
                expiration = datetime.strptime(expiration_raw, "%Y%m%d")


        # =====================================
        # FILTRO DE TICKER REAL DE OPÇÃO
        # =====================================

        if pd.notna(strike):

            # elimina séries fantasmas tipo:
            # PETRD540W5
            # PETRD540M
            # PETRD540T
            # etc

            if not OPTION_REGEX.match(ticker):
                continue


        records.append({

            "ticker": ticker,
            "fechamento_d-1": close,
            "volume_d-1": volume,
            "strike": strike,
            "expiration": expiration

        })

    return pd.DataFrame(records)

# =========================================
# EXECUÇÃO
# =========================================

trade_date, lines = download_latest_cotahist()

df_raw = parse_cotahist(lines, UNDERLYINGS_BASE)

print("Data de referência:", trade_date)
print("Qtd registros após filtro base:", len(df_raw))

display(df_raw.head(20))

✅ Arquivo encontrado: https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_D18032026.ZIP
Arquivo interno: COTAHIST_D18032026.TXT
Total de linhas: 17591
Data de referência: 2026-03-18
Qtd registros após filtro base: 3140


,ticker,fechamento_d-1,volume_d-1,strike,expiration
0,SUZB3,52.56,8819400,NaN,9999-12-31
1,BBAS3,23.41,15566500,NaN,9999-12-31
2,PETR3,51.63,16908500,NaN,9999-12-31
3,PETR4,47.00,55074700,NaN,9999-12-31
4,B3SA3,17.12,23216900,NaN,9999-12-31
5,BBDC3,16.08,6181400,NaN,9999-12-31
6,BBDC4,18.63,26382300,NaN,9999-12-31
7,WEGE3,46.16,5688200,NaN,9999-12-31
8,VALE3,77.13,19412200,NaN,9999-12-31
9,ABEV3,14.74,20403700,NaN,9999-12-31


In [9]:
# %%
from datetime import datetime, timedelta

# =========================================
# 3ª SEXTA
# =========================================

def third_friday(year, month):
    d = datetime(year, month, 1)
    days_until_friday = (4 - d.weekday()) % 7
    first_friday = d + timedelta(days=days_until_friday)
    third = first_friday + timedelta(days=14)
    return third.date()


# usa a data do arquivo, não o relógio do PC
now = datetime.combine(trade_date, datetime.min.time())

current_expiry_date = third_friday(now.year, now.month)

if now.month == 12:
    next_month = 1
    next_year = now.year + 1
else:
    next_month = now.month + 1
    next_year = now.year

next_expiry_date = third_friday(next_year, next_month)

print("Vencimento mês atual :", current_expiry_date)
print("Vencimento próximo mês:", next_expiry_date)


# =========================================
# CLASSIFICAÇÃO
# =========================================

def classify_asset(ticker, strike):
    if pd.notna(strike):
        letter = ticker[4]

        if letter in list("ABCDEFGHIJKL"):
            return "CALL"
        elif letter in list("MNOPQRSTUVWX"):
            return "PUT"
        else:
            return "OPTION"

    return "SPOT"


df_raw["type"] = df_raw.apply(
    lambda r: classify_asset(r["ticker"], r["strike"]),
    axis=1
)


# =========================================
# UNDERLYING CORRETO
# =========================================

def infer_underlying(ticker, asset_type):
    base = ticker[:4]

    if asset_type == "SPOT":
        return ticker

    return UNDERLYING_MAP.get(base)


df_raw["underlying"] = df_raw.apply(
    lambda r: infer_underlying(r["ticker"], r["type"]),
    axis=1
)

print(df_raw["type"].value_counts(dropna=False))
display(df_raw.head(20))

Vencimento mês atual : 2026-03-20
Vencimento próximo mês: 2026-04-17
type
CALL    1516
PUT     1235
SPOT     389
Name: count, dtype: int64


,ticker,fechamento_d-1,volume_d-1,strike,expiration,type,underlying
0,SUZB3,52.56,8819400,NaN,9999-12-31,SPOT,SUZB3
1,BBAS3,23.41,15566500,NaN,9999-12-31,SPOT,BBAS3
2,PETR3,51.63,16908500,NaN,9999-12-31,SPOT,PETR3
3,PETR4,47.00,55074700,NaN,9999-12-31,SPOT,PETR4
4,B3SA3,17.12,23216900,NaN,9999-12-31,SPOT,B3SA3
5,BBDC3,16.08,6181400,NaN,9999-12-31,SPOT,BBDC3
6,BBDC4,18.63,26382300,NaN,9999-12-31,SPOT,BBDC4
7,WEGE3,46.16,5688200,NaN,9999-12-31,SPOT,WEGE3
8,VALE3,77.13,19412200,NaN,9999-12-31,SPOT,VALE3
9,ABEV3,14.74,20403700,NaN,9999-12-31,SPOT,ABEV3


In [10]:
# %%
# =========================================
# UNIVERSO IGUAL AO MT5
# =========================================

df_spot = df_raw[df_raw["type"] == "SPOT"].copy()

df_calls = df_raw[
    (df_raw["type"] == "CALL") &
    (df_raw["expiration"].notna()) &
    (df_raw["expiration"].dt.date.isin([current_expiry_date, next_expiry_date]))
].copy()

print("SPOT:", len(df_spot))
print("CALL:", len(df_calls))


# =========================================
# PREÇO DO SPOT NAS CALLs
# =========================================

spot_price_map = dict(zip(df_spot["ticker"], df_spot["fechamento_d-1"]))
df_calls["spot_price"] = df_calls["underlying"].map(spot_price_map)

print("CALLs com spot mapeado:", df_calls["spot_price"].notna().sum())
display(df_calls.head(20))


# =========================================
# CALLs OTM
# =========================================

df_calls_valid = df_calls[
    (df_calls["strike"].notna()) &
    (df_calls["spot_price"].notna()) &
    (df_calls["strike"] > df_calls["spot_price"])
].copy()

print("CALLs OTM:", len(df_calls_valid))

display(
    df_calls_valid[
        ["ticker", "underlying", "spot_price", "strike", "expiration", "fechamento_d-1", "volume_d-1"]
    ].head(20)
)


# =========================================
# GERAR CALL CREDIT SPREADS
# =========================================

spreads = []

group_cols = ["underlying", "expiration"]

for (underlying, expiration), g in df_calls_valid.groupby(group_cols):

    g = g.sort_values("strike")

    for _, sell in g.iterrows():

        for _, buy in g.iterrows():

            if buy["strike"] <= sell["strike"]:
                continue

            credit = sell["fechamento_d-1"] - buy["fechamento_d-1"]

            if credit <= 0:
                continue

            spreads.append({

                "underlying": underlying,
                "expiration": expiration,

                # TICKERS REAIS
                "sell_ticker": sell["ticker"],
                "buy_ticker": buy["ticker"],

                "sell_strike": sell["strike"],
                "buy_strike": buy["strike"],

                "spot_price": sell["spot_price"],

                "sell_premium": sell["fechamento_d-1"],
                "buy_premium": buy["fechamento_d-1"],
                "credit": credit,

                "sell_volume": sell["volume_d-1"],
                "buy_volume": buy["volume_d-1"],
            })

df_call_spreads = pd.DataFrame(spreads)

print("Qtd spreads gerados:", len(df_call_spreads))
display(df_call_spreads.head(20))

SPOT: 389
CALL: 930
CALLs com spot mapeado: 930


,ticker,fechamento_d-1,volume_d-1,strike,expiration,type,underlying,spot_price
22,ABEVC111,4.61,2600,10.44,2026-03-20,CALL,ABEV3,14.74
23,ABEVC122,3.52,5000,11.44,2026-03-20,CALL,ABEV3,14.74
24,ABEVC123,3.24,1700,11.69,2026-03-20,CALL,ABEV3,14.74
25,ABEVC128,2.76,12200,12.19,2026-03-20,CALL,ABEV3,14.74
26,ABEVC131,2.40,24700,12.44,2026-03-20,CALL,ABEV3,14.74
27,ABEVC133,2.16,18600,12.69,2026-03-20,CALL,ABEV3,14.74
28,ABEVC136,1.90,10600,12.94,2026-03-20,CALL,ABEV3,14.74
29,ABEVC138,1.73,46100,13.19,2026-03-20,CALL,ABEV3,14.74
31,ABEVC141,1.38,252200,13.44,2026-03-20,CALL,ABEV3,14.74
32,ABEVC143,1.14,13400,13.69,2026-03-20,CALL,ABEV3,14.74


CALLs OTM: 356


,ticker,underlying,spot_price,strike,expiration,fechamento_d-1,volume_d-1
38,ABEVC156,ABEV3,14.74,14.94,2026-03-20,0.11,601400
39,ABEVC158,ABEV3,14.74,15.19,2026-03-20,0.04,355800
40,ABEVC161,ABEV3,14.74,15.44,2026-03-20,0.02,144300
41,ABEVC163,ABEV3,14.74,15.69,2026-03-20,0.02,12500
42,ABEVC166,ABEV3,14.74,15.94,2026-03-20,0.01,1100
43,ABEVC173,ABEV3,14.74,16.69,2026-03-20,0.01,300
57,ABEVD149,ABEV3,14.74,14.92,2026-04-17,0.48,376100
58,ABEVD151,ABEV3,14.74,15.17,2026-04-17,0.36,723600
59,ABEVD154,ABEV3,14.74,15.42,2026-04-17,0.27,190900
60,ABEVD156,ABEV3,14.74,15.67,2026-04-17,0.18,128400


Qtd spreads gerados: 4914


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume
0,ABEV3,2026-03-20,ABEVC156,ABEVC158,14.94,15.19,14.74,0.11,0.04,0.07,601400,355800
1,ABEV3,2026-03-20,ABEVC156,ABEVC161,14.94,15.44,14.74,0.11,0.02,0.09,601400,144300
2,ABEV3,2026-03-20,ABEVC156,ABEVC163,14.94,15.69,14.74,0.11,0.02,0.09,601400,12500
3,ABEV3,2026-03-20,ABEVC156,ABEVC166,14.94,15.94,14.74,0.11,0.01,0.10,601400,1100
4,ABEV3,2026-03-20,ABEVC156,ABEVC173,14.94,16.69,14.74,0.11,0.01,0.10,601400,300
5,ABEV3,2026-03-20,ABEVC158,ABEVC161,15.19,15.44,14.74,0.04,0.02,0.02,355800,144300
6,ABEV3,2026-03-20,ABEVC158,ABEVC163,15.19,15.69,14.74,0.04,0.02,0.02,355800,12500
7,ABEV3,2026-03-20,ABEVC158,ABEVC166,15.19,15.94,14.74,0.04,0.01,0.03,355800,1100
8,ABEV3,2026-03-20,ABEVC158,ABEVC173,15.19,16.69,14.74,0.04,0.01,0.03,355800,300
9,ABEV3,2026-03-20,ABEVC161,ABEVC166,15.44,15.94,14.74,0.02,0.01,0.01,144300,1100


In [11]:
# %%
# =========================================
# PARÂMETROS
# =========================================

MIN_SELL_VOLUME    = 5000
MIN_BUY_VOLUME     = 1000
MIN_OTM_PCT        = 2.0
MIN_RETURN_PCT     = 30.0
MIN_CREDIT         = 0.5

MIN_SPREAD_WIDTH   = 1.0
MAX_SPREAD_WIDTH   = 2.0


# =========================================
# MÉTRICAS (IGUAL AO MT5)
# =========================================

df = df_call_spreads.copy()

df["spread_width"] = df["buy_strike"] - df["sell_strike"]

df["otm_pct"] = (
    (df["sell_strike"] - df["spot_price"]) / df["spot_price"]
) * 100

df["max_loss"] = df["spread_width"] - df["credit"]

df["return_pct"] = (df["credit"] / df["max_loss"]) * 100


# =========================================
# FILTROS
# =========================================

df_filt = df[
    (df["sell_volume"]   >= MIN_SELL_VOLUME)   &
    (df["buy_volume"]    >= MIN_BUY_VOLUME)    &
    (df["otm_pct"]       >= MIN_OTM_PCT)       &
    (df["return_pct"]    >= MIN_RETURN_PCT)    &
    (df["credit"]        >= MIN_CREDIT)        &
    (df["spread_width"]  >= MIN_SPREAD_WIDTH)  &
    (df["spread_width"]  <= MAX_SPREAD_WIDTH)  &
    (df["max_loss"]      > 0)
].copy()

print("Qtd travas após filtros:", len(df_filt))


# =========================================
# SEPARA VENCIMENTOS
# =========================================

df_filt["exp_date"] = df_filt["expiration"].dt.date

df_current = df_filt[df_filt["exp_date"] == current_expiry_date]
df_next    = df_filt[df_filt["exp_date"] == next_expiry_date]


# =========================================
# EXIBIÇÃO
# =========================================

def show_table(df_input, titulo):
    print("")
    print("================================")
    print(titulo)
    print("================================")

    if df_input.empty:
        print("Nenhuma trava encontrada")
        return

    df_rank = df_input.sort_values(
        ["return_pct", "otm_pct"],
        ascending=[False, False]
    )

    display(df_rank.head(20))


show_table(df_current, "TRAVAS — VENCIMENTO MÊS ATUAL")
show_table(df_next, "TRAVAS — VENCIMENTO PRÓXIMO MÊS")

Qtd travas após filtros: 34

TRAVAS — VENCIMENTO MÊS ATUAL
Nenhuma trava encontrada

TRAVAS — VENCIMENTO PRÓXIMO MÊS


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume,spread_width,otm_pct,max_loss,return_pct,exp_date
1586,PETR4,2026-04-17,PETRD485,PETRD495,48.50,49.50,47.00,4.93,4.23,0.70,10200,13200,1.00,3.191489,0.30,233.333333,2026-04-17
4627,WEGE3,2026-04-17,WEGED488,WEGED490,47.61,49.11,46.16,1.45,0.82,0.63,19800,13100,1.50,3.141248,0.87,72.413793,2026-04-17
1510,PETR4,2026-04-17,PETRD481,PETRD496,48.15,49.65,47.00,1.90,1.31,0.59,3741900,262600,1.50,2.446809,0.91,64.835165,2026-04-17
1511,PETR4,2026-04-17,PETRD481,PETRD499,48.15,49.90,47.00,1.90,1.23,0.67,3741900,218700,1.75,2.446809,1.08,62.037037,2026-04-17
4629,WEGE3,2026-04-17,WEGED488,WEGED496,47.61,49.61,46.16,1.45,0.70,0.75,19800,9700,2.00,3.141248,1.25,60.000000,2026-04-17
2456,SUZB3,2026-04-17,SUZBD538,SUZBD553,53.88,55.38,52.56,1.52,0.97,0.55,35200,13700,1.50,2.511416,0.95,57.894737,2026-04-17
4577,WEGE3,2026-04-17,WEGED483,WEGED498,47.11,48.61,46.16,1.54,0.99,0.55,18200,14500,1.50,2.058059,0.95,57.894737,2026-04-17
3109,VALE3,2026-04-17,VALED789,VALED804,78.90,80.40,77.13,2.26,1.71,0.55,631900,759300,1.50,2.294827,0.95,57.894737,2026-04-17
1512,PETR4,2026-04-17,PETRD481,PETRD501,48.15,50.15,47.00,1.90,1.17,0.73,3741900,604300,2.00,2.446809,1.27,57.480315,2026-04-17
4579,WEGE3,2026-04-17,WEGED483,WEGED490,47.11,49.11,46.16,1.54,0.82,0.72,18200,13100,2.00,2.058059,1.28,56.250000,2026-04-17


In [12]:
# =========================================
# TELEGRAM
# =========================================

import requests

TELEGRAM_BOT_TOKEN = "8363974523:AAGGmbw7gfIUlCnjkYVSBLKXbiShQhA5yGg"
TELEGRAM_CHAT_ID = "8564196952"


def send_telegram_message(text):

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": text,
        "parse_mode": "Markdown"
    }

    requests.post(url, json=payload)

In [13]:
def format_spread_message(row):

    msg = f"""
📊 *Trava de Crédito Detectada*

{row.sell_ticker} / {row.buy_ticker}

Spot: {row.spot_price:.2f}

SELL strike: {row.sell_strike:.2f}
BUY strike:  {row.buy_strike:.2f}

Crédito: {row.credit:.2f}
Risco:   {row.max_loss:.2f}

Width: {row.spread_width:.2f}

OTM: {row.otm_pct:.2f}%
Retorno: {row.return_pct:.2f}%
"""

    return msg

In [16]:
# =========================================
# TELEGRAM ALERT — BLOOMBERG STYLE
# =========================================

df_rank_next = df_filt[
    df_filt["exp_date"] == next_expiry_date
].sort_values(
    ["return_pct", "otm_pct"],
    ascending=[False, False]
)

top_spreads = df_rank_next.head(10)

msg = "📊 *TOP 10 TRAVAS - PRÓX VCTO*\n\n"
msg += "`SELL/BUY        OTM   CR   RET`\n"

for row in top_spreads.itertuples():

    sell = row.sell_ticker
    buy  = row.buy_ticker

    msg += (
        f"`{sell}/{buy[-3:]} "
        f"{row.otm_pct:>5.2f} "
        f"{row.credit:>4.2f} "
        f"{row.return_pct:>4.0f}`\n"
    )

send_telegram_message(msg)